# Module 3: Build an Annual Financial Panel

目标：将 AEO、ANF、URBN 的原始财务记录整理成**每家公司、每个实际年度期间一行**的工作表。
沿用基础语法：普通循环、字典、筛选、排序和保存。不用 `lambda` 或自定义处理函数。
教学说明用中文，代码及运行提示用英文。按 **Shift + Enter** 顺序运行。

本模块读取模块 1 的 CSV 和模块 2 的三份 Company Facts 缓存；**不发出网络请求，不需要邮箱**。
可在新的 Kernel 中运行。请先确认模块 2 已保存三家公司的 JSON，并使用相同的项目目录。
本机仅以模拟数据检查流程，没有真实缓存可以预填结果。

工作表包含收入、销售成本、毛利额、存货、资产和经营现金流。
标签选择是待审核的初步规则；年度表还需要在模块 4 对照年报、检查缺失和口径。
已有缓存若缺少候选标签，本模块不会编造金额，也不会自动下载补充。

三个处理原则：
1. 用 `start` 和 `end` 判断实际期间，不用 `fy` 把比较年度归到提交年报的年份。
2. 每个年度先选择缓存中最早可见的全年收入年报记录；其他项目匹配同一份文件。
3. 缺失保留为空；同一标签、同一文件、同一期间的金额发生冲突时也留空，待人工核对。

这是原始披露口径的年度工作表，**不是“所有历史时点完全无偏”的回测数据认证**。
“最早”指当前缓存里最早的符合条件记录，不保证是该年度首次对外公布的数字。

来源：[SEC XBRL API 说明](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)。


## Cell 1: Install Pandas

本模块只需要 pandas 和 Python 自带工具。前面安装过时可跳过。


In [ ]:
%pip install pandas


## Cell 2: Set Up the Project

项目路径必须与模块 1、2 一致。`FACTS_DIR` 指向已有缓存，`PROCESSED_DIR` 指向整理数据目录。
本格只导入工具和确定路径，不修改原始 JSON。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
FACTS_DIR = PROJECT_ROOT / "work" / "data" / "raw" / "companyfacts"
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
print("Project directory:", PROJECT_ROOT)


## Cell 3: Read the Three Cached Snapshots

读取模块 1 的公司清单，找到三家公司的 CIK，再逐个打开模块 2 保存的 JSON。
`company_facts` 保存公司财务对象；`snapshot_times` 记录各份缓存的获取时间。
如果文件不存在，请先核对路径并完成模块 2，不要手工填入财务数值。


In [ ]:
universe = pd.read_csv(PROCESSED_DIR / "retail_candidate_universe.csv", dtype={"cik10": "string"})
pilot = universe.loc[universe["ticker"].isin(["AEO", "ANF", "URBN"])]
if len(pilot) != 3 or pilot["cik10"].isna().any():
    raise ValueError("The pilot companies or CIKs are missing. Check Module 1.")

company_facts = {}
snapshot_times = {}
for index, row in pilot.iterrows():
    cache_path = FACTS_DIR / f"CIK{row['cik10']}_companyfacts.json"
    snapshot = json.loads(cache_path.read_text(encoding="utf-8"))
    if int(snapshot["payload"]["cik"]) != int(row["cik10"]):
        raise ValueError(f"{row['ticker']}: CIK mismatch in the cache.")
    company_facts[row["ticker"]] = snapshot["payload"]
    snapshot_times[row["ticker"]] = snapshot["retrieved_at_utc"]
    print(row["ticker"], "loaded", snapshot["retrieved_at_utc"])


## Cell 4: Collect the Financial Records

像模块 2 一样定义候选标签，但本次把具体金额、期间和来源也取出，形成一张“长表”：每行是一条披露记录。
额外保留 `GrossProfit`，因为各公司的销售成本费用范围不同，不能预先保证“收入减选出的销售成本”等于报表毛利。
原始 Company Facts 已含所有可取得的标准标签，不必为这个新增项目重新下载。

三层循环依次处理公司、指标、候选标签，然后逐条读取该标签的 USD 记录。
`enumerate(possible_tags)` 给标签编号 0、1、2……，作为**暂定选择顺序**，不是已验证的会计优劣排序。
`fact.get("start")` 允许时点项目没有起始日期；缺失保留为空。

`pd.to_datetime` 将日期文字转为日期对象，后面才能计算期间长度。
只删除所有所列来源字段与金额都相同的重复记录，保留不同文件、不同标签、不同金额的披露。
金额仍为原始美元单位，不进行百万元缩放。


In [ ]:
tag_candidates = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet", "Revenues", "SalesRevenueGoodsNet"],
    "cost_of_sales": ["CostOfRevenue", "CostOfGoodsAndServicesSold", "CostOfGoodsSold"],
    "gross_profit": ["GrossProfit"],
    "inventory": ["InventoryNet"],
    "total_assets": ["Assets"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities", "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"],
}
rows = []
for ticker, data in company_facts.items():
    tags = data["facts"].get("us-gaap", {})
    for metric, possible_tags in tag_candidates.items():
        for priority, tag in enumerate(possible_tags):
            records = tags.get(tag, {}).get("units", {}).get("USD", [])
            for fact in records:
                rows.append({
                    "ticker": ticker, "metric": metric, "tag": tag, "priority": priority,
                    "start": fact.get("start"), "end": fact["end"], "value": fact["val"],
                    "form": fact["form"], "filed": fact["filed"], "accn": fact["accn"],
                })

facts = pd.DataFrame(rows)
if facts.empty:
    raise ValueError("No candidate USD records found. Review Module 2 and the annual reports.")
for column in ["start", "end", "filed"]:
    facts[column] = pd.to_datetime(facts[column])
facts = facts.drop_duplicates()
display(facts.head())


## Cell 5: Identify Annual Periods

收入是期间值，因此先用它建立年度期间。
`end - start` 得到两个日期的间隔，`.dt.days + 1` 将起止日都计入。
筛选 10-K/10-K/A 中长度为 **350–380 天**的收入记录，覆盖通常的 52/53 周或自然年度。
这个范围是教学用规则；短过渡财年会被排除，后续需要专门审核，不能据此认定公司没有该年数据。

先按披露时间排序，再按公司和期间终点去重，保留最早可见的全年收入记录，命名为 `anchors`。
同一提交文件内有多个候选收入标签时，使用暂定标签顺序。
10-K 比较数字可能对应更早的期间，因此不用 `fy` 或简单的 `end.year` 命名公司财年。

`filed` 是文件提交时间，不是财年结束时间或业绩公告时间；同一份文件用 `accn` 识别。
这条规则避免直接把后来年报中的比较值当成当时已经可用的数值，但不能证明缓存包含所有最初披露版本。


In [ ]:
facts["period_days"] = (facts["end"] - facts["start"]).dt.days + 1
annual_revenue = facts.loc[
    (facts["metric"] == "revenue")
    & facts["form"].isin(["10-K", "10-K/A"])
    & facts["period_days"].between(350, 380)
].copy()
anchors = annual_revenue.sort_values(["ticker", "end", "filed", "accn", "priority"])
anchors = anchors.drop_duplicates(["ticker", "end"], keep="first")
if anchors.empty:
    raise ValueError("No full-year revenue periods found. Review the revenue tags.")
display(anchors[["ticker", "start", "end", "period_days", "filed", "accn", "tag"]].tail(10))


## Cell 6: Build One Row per Annual Period

这一格是主要整理步骤，分成“每个年度”和“每个指标”两个普通循环。

1. 从 `anchors` 取出一家公司的实际年度起止日期、提交时间和文件编号。
2. 每个指标匹配同公司、同期间终点、同文件编号的记录。
3. 存货和资产是时点值，只要求日期等于年度终点；其他四个期间值还要求起始日期与年度收入一致。
4. 按候选标签顺序取一个标签。`.iloc[0]` 取排序后的第一行，并非随意取最新披露。
5. 选中标签在同一文件、同一期间的金额唯一时才填值；冲突标记 `ambiguous`，没有记录标记 `missing`。

`annual_row[metric] = value` 逐个增加财务列；每个年度结束时把这一行放进列表。
`field_sources` 另存指标来源和处理状态，方便模块 4 核对，不用把所有来源列塞进主表。
`selected_pending_review` 表示程序按规则选中了值，**不表示会计口径审核已经完成**。

多个候选标签的费用或合并范围可能不同。暂定顺序必须对照年报核实，不能直接把各公司的毛利率水平当成可比排名。
同一文件中另一个标签有不同金额时，不自动相加；原始候选记录仍在 `facts` 和 JSON 中可查。


In [ ]:
annual_rows = []
source_rows = []
for index, year in anchors.iterrows():
    annual_row = {
        "ticker": year["ticker"], "period_start": year["start"], "period_end": year["end"],
        "period_days": year["period_days"], "filed": year["filed"], "accn": year["accn"],
    }
    for metric in tag_candidates:
        matches = facts.loc[
            (facts["ticker"] == year["ticker"]) & (facts["metric"] == metric)
            & (facts["end"] == year["end"]) & (facts["accn"] == year["accn"])
        ]
        if metric not in ["inventory", "total_assets"]:
            matches = matches.loc[matches["start"] == year["start"]]
        value = None
        tag = None
        status = "missing"
        if not matches.empty:
            matches = matches.sort_values("priority")
            tag = matches.iloc[0]["tag"]
            chosen = matches.loc[matches["tag"] == tag]
            if chosen["value"].nunique() == 1:
                value = chosen.iloc[0]["value"]
                status = "selected_pending_review"
            else:
                status = "ambiguous"
        annual_row[metric] = value
        source_rows.append({
            "ticker": year["ticker"], "period_end": year["end"], "metric": metric,
            "tag": tag, "value": value, "status": status, "accn": year["accn"],
        })
    annual_rows.append(annual_row)

annual_panel = pd.DataFrame(annual_rows).sort_values(["ticker", "period_end"])
field_sources = pd.DataFrame(source_rows)
display(annual_panel.head())


## Cell 7: Preview the Annual Panel

展示 AEO 的最后五个年度期间、各指标缺失行数，以及需要核对的字段来源。
这里不填 0、不删除不完整年度，也不宣称已经有多少个合格研究样本。
`period_end` 是实际年度终点，不是三家公司统一的“Fiscal Year”标签。
若某列大量缺失，请先查模块 2 的标签概览、`facts` 和年报，再修改标签规则重新运行。


In [ ]:
display(annual_panel.loc[annual_panel["ticker"] == "AEO"].tail(5))
metrics = list(tag_candidates.keys())
display(annual_panel[metrics].isna().sum().to_frame("Missing Rows"))
display(field_sources.loc[field_sources["status"] != "selected_pending_review"].head(10))
print("Annual period rows:", len(annual_panel))


## Cell 8: Save the Annual Panel and Sources

保存年度主表、逐字段来源表、规则及快照时间说明。再次运行会覆盖同名文件，原始 Company Facts 缓存不变。
CSV 中的日期统一写成 `YYYY-MM-DD`；下个模块读回时需要用 `pd.to_datetime` 恢复日期类型。
metadata 中记录候选标签顺序和未完成口径审核的状态，让处理规则可以追溯。


In [ ]:
panel_path = PROCESSED_DIR / "pilot_annual_panel.csv"
sources_path = PROCESSED_DIR / "pilot_annual_field_sources.csv"
annual_panel.to_csv(panel_path, index=False, date_format="%Y-%m-%d")
field_sources.to_csv(sources_path, index=False, date_format="%Y-%m-%d")
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "snapshot_times": snapshot_times, "unit": "USD", "annual_period_days": [350, 380],
    "filing_rule": "Earliest available full-year revenue 10-K/10-K/A per period end; other metrics use the same accession.",
    "tag_candidates_in_priority_order": tag_candidates,
    "accounting_scope_verified": False,
}
metadata_path = PROCESSED_DIR / "pilot_annual_panel_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved annual panel:", panel_path)
print("Saved field sources:", sources_path)


## Completion and Next Step

完成后应有年度工作表、字段来源表和 metadata；实际年度数量以你的缓存运行结果为准。
本模块不计算增长率、库存信号或投资结论。模块 4 先核对口径与缺失，再考虑扩大公司样本。

用于人工核对的公开年报：
- [AEO 2026 年报文件](https://www.sec.gov/Archives/edgar/data/919012/000119312526132097/aeo-20260131.htm)
- [ANF 2025 财年年报](https://www.sec.gov/Archives/edgar/data/1018840/000101884026000012/anf-20260131.htm)
- [URBN 2026 财年年报](https://www.sec.gov/Archives/edgar/data/912615/000119312526137916/urbn-20260131.htm)

AEO 的销售成本包含部分采购、租赁和仓储费用；ANF 的披露排除折旧摊销；
URBN 的毛利计算还涉及单独列示的门店减值及租赁放弃费用。因此需要保留直接披露的毛利额并审核费用范围。
URBN 还包含批发和订阅业务；当前整理的是公司整体数据，最终是否纳入零售研究仍需业务审核。

只要缺失或冲突如实显示，表格仍有助于排查；不要为了让程序显示完整行而补 0 或任意选择不同文件的数据。
